# LSTM-Only Baseline — IBM & Sparkov
Standalone notebook. Runs **only** the pure LSTM baseline (`lstm_solo/`) —
zero graph component, no `edge_index`, no GAT layer anywhere. This is
kept entirely separate from the hybrid GAT+LSTM architectures (which live
in `ibm/`/`sparkov/` and are run by `run_all_colab.ipynb` instead).

Purpose: isolate what the LSTM branch alone achieves, with no relational
signal, for direct comparison against the GATv2-only baseline reported in
the hybrid notebook — this answers whether the graph contributes anything
on top of temporal modelling alone.

| File | Node granularity | Graph strategies |
|---|---|---|
| `lstm_solo/ibm/lstm_only_model.py` | Transaction | none — runs once, not per-strategy |
| `lstm_solo/sparkov/lstm_only_model.py` | Transaction | none — runs once, not per-strategy |

---
### Before you start
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. Update `REPO_URL` in Cell 2
3. Run the IBM section, the Sparkov section, or both — independent, no shared state

---
## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
import torch

torch_version = torch.__version__.split('+')[0]
cuda_version  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_version}  |  CUDA: {cuda_version}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'pandas', 'matplotlib', 'pyarrow'], check=True)
print('Done.')

---
## Cell 2 — Clone the repo
This clones the whole repo (LSTM-only lives inside it at `lstm_solo/`),
but this notebook never touches `ibm/` or `sparkov/`.

In [ ]:
import os

REPO_URL = 'https://github.com/<your-username>/hybrid-gnn-lstm-fraud.git'  # ← update

if not os.path.isdir('/content/hybrid-gnn-lstm-fraud'):
    !git clone -q {REPO_URL} /content/hybrid-gnn-lstm-fraud

os.makedirs('/content/outcomes', exist_ok=True)
print('Repo ready at /content/hybrid-gnn-lstm-fraud')

---
## Cell 2b — Mount Google Drive
Required: results are saved here incrementally so a runtime disconnect
mid-run doesn't lose completed work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
# Part A — IBM

## Cell 3 — Load and preprocess
Set `IBM_DATA_PATH` to your actual `reduced_dataset.parquet` location.

In [ ]:
import os, sys

IBM_DATA_PATH = '/content/drive/MyDrive/reduced_dataset.parquet'  # ← update

os.chdir('/content/hybrid-gnn-lstm-fraud/lstm_solo/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes/ibm'

import utils as ibm_utils
df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)
print(f'{len(df_ibm):,} transactions loaded.')

---
## Cell 4 — IBM: LSTM-only baseline
No `graph_strategy` loop — this model has no graph component, so it
runs once (not three times) using the same 5-fold splits as the hybrid
notebook's models. ~10–20 min on T4 GPU. Saves to Drive; re-running this
cell after a disconnect skips straight to the cached result.

In [ ]:
import lstm_only_model as ibm_lstm_only
import pickle

save_path = '/content/drive/MyDrive/ibm_lstm_only_result.pkl'
if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        lstm_only_results_ibm = pickle.load(f)
    print("Skipping — already saved.")
else:
    lstm_only_results_ibm = ibm_lstm_only.run_all_strategies(df_ibm, cfg_ibm)
    with open(save_path, 'wb') as f:
        pickle.dump(lstm_only_results_ibm, f)
    print("Saved to Drive.")

print('\nIBM LSTM-only done.')

import pandas as pd
m = lstm_only_results_ibm['no_topology_lstm_only']['test_metrics']
print(f"TEST — F1 {m['f1']:.4f} | Prec {m['prec']:.4f} | Rec {m['rec']:.4f} | AUC {m['auc']:.4f} | AP {m['ap']:.4f}")

---
# Part B — Sparkov

## Cell 5 — Load and preprocess
Set `SPARKOV_TRAIN_PATH` / `SPARKOV_TEST_PATH` to your actual
`fraudTrain.csv` / `fraudTest.csv` location. Independent of Part A —
safe to run this section on its own.

In [ ]:
import os, sys

SPARKOV_TRAIN_PATH = '/content/fraudTrain.csv'  # ← update if needed
SPARKOV_TEST_PATH  = '/content/fraudTest.csv'   # ← update if needed

os.chdir('/content/hybrid-gnn-lstm-fraud/lstm_solo/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg_sparkov = sparkov_cfg.CardFraudConfig()
cfg_sparkov.OUTCOME_DIR = '/content/outcomes/sparkov'

import utils as sparkov_utils
df_sparkov = sparkov_utils.load_and_preprocess(
    train_path=SPARKOV_TRAIN_PATH, test_path=SPARKOV_TEST_PATH, cfg=cfg_sparkov)
print(f'{len(df_sparkov):,} transactions loaded.')

---
## Cell 6 — Sparkov: LSTM-only baseline
Same ablation as Cell 4. Runs once (no `graph_strategy` loop).
~10–20 min on T4 GPU. Saves to Drive.

In [ ]:
import lstm_only_model as sparkov_lstm_only
import pickle

save_path = '/content/drive/MyDrive/sparkov_lstm_only_result.pkl'
if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        lstm_only_results_sparkov = pickle.load(f)
    print("Skipping — already saved.")
else:
    lstm_only_results_sparkov = sparkov_lstm_only.run_all_strategies(df_sparkov, cfg_sparkov)
    with open(save_path, 'wb') as f:
        pickle.dump(lstm_only_results_sparkov, f)
    print("Saved to Drive.")

print('\nSparkov LSTM-only done.')

m = lstm_only_results_sparkov['no_topology_lstm_only']['test_metrics']
print(f"TEST — F1 {m['f1']:.4f} | Prec {m['prec']:.4f} | Rec {m['rec']:.4f} | AUC {m['auc']:.4f} | AP {m['ap']:.4f}")

---
## Cell 7 — Combined summary (both datasets, if both were run)

In [ ]:
import pandas as pd

rows = []
try:
    m = lstm_only_results_ibm['no_topology_lstm_only']['test_metrics']
    rows.append({'dataset': 'IBM', 'f1': round(m['f1'],4), 'prec': round(m['prec'],4),
                 'rec': round(m['rec'],4), 'auc': round(m['auc'],4), 'ap': round(m['ap'],4)})
except NameError:
    print('IBM not run in this session — skipping.')

try:
    m = lstm_only_results_sparkov['no_topology_lstm_only']['test_metrics']
    rows.append({'dataset': 'Sparkov', 'f1': round(m['f1'],4), 'prec': round(m['prec'],4),
                 'rec': round(m['rec'],4), 'auc': round(m['auc'],4), 'ap': round(m['ap'],4)})
except NameError:
    print('Sparkov not run in this session — skipping.')

df_summary = pd.DataFrame(rows)
os.makedirs('/content/outcomes', exist_ok=True)
df_summary.to_csv('/content/outcomes/lstm_only_summary.csv', index=False)
df_summary

---
## Cell 8 — Download results

In [ ]:
!cd /content/outcomes && zip -qr /content/outcomes_lstm_only.zip .
from google.colab import files
files.download('/content/outcomes_lstm_only.zip')